# Cell 01 - Load dữ liệu cho bước preprocessing và chunking

## Mục tiêu cell này
Đọc lại các dữ liệu đã chuẩn bị từ notebook 01 và 02 để bắt đầu tiền xử lý văn bản.

## Giải thích code
Code sẽ load:
- `legal_corpus.csv`: kho văn bản pháp luật tiếng Việt
- `company_handbook_documents.csv`: tài liệu nội bộ công ty 37signals
- `train_strict.csv`, `val_strict.csv`, `test_final.csv`: các split đã kiểm soát leakage

Sau đó in shape để kiểm tra dữ liệu đã sẵn sàng cho bước chunking.

## Output mong đợi
Bạn cần thấy:
- legal corpus khoảng 68,663 dòng
- company handbook 16 dòng
- train/val/test đúng với split strict đã tạo ở notebook 02

In [1]:
from pathlib import Path
import pandas as pd
import json
import re

cwd = Path.cwd().resolve()
PROJECT = cwd.parent if cwd.name == "notebooks" else cwd

PROCESSED_DIR = PROJECT / "data" / "processed"
SPLIT_DIR = PROJECT / "data" / "splits"
CHUNK_DIR = PROJECT / "data" / "chunks"

CHUNK_DIR.mkdir(parents=True, exist_ok=True)

legal_corpus_df = pd.read_csv(PROCESSED_DIR / "legal_corpus.csv")
company_docs_df = pd.read_csv(PROCESSED_DIR / "company_handbook_documents.csv")

train_df = pd.read_csv(SPLIT_DIR / "train_strict.csv")
val_df = pd.read_csv(SPLIT_DIR / "val_strict.csv")
test_df = pd.read_csv(SPLIT_DIR / "test_final.csv")

train_df["relevant_cids"] = train_df["relevant_cids"].apply(json.loads)
val_df["relevant_cids"] = val_df["relevant_cids"].apply(json.loads)
test_df["relevant_cids"] = test_df["relevant_cids"].apply(json.loads)

print("Project:", PROJECT)
print("Legal corpus:", legal_corpus_df.shape)
print("Company handbook:", company_docs_df.shape)
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

display(legal_corpus_df.head())
display(company_docs_df[["doc_id", "title", "text_length", "source_type"]].head())

Project: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag
Legal corpus: (68663, 3)
Company handbook: (16, 6)
Train: (78812, 4)
Validation: (8605, 4)
Test: (29746, 4)


,cid,context_text,source_type
0,142820,“Điều 2. Địa vị pháp lý của Liên đoàn Luật sư ...,legal
1,27817,"""Điều 7. Tên hợp tác xã, liên hiệp hợp tác xã\...",legal
2,72117,"Cơ quan đăng ký hợp tác xã\n1. Khi thành lập, ...",legal
3,33215,"""1. Sử dụng lái xe bảo đảm sức khỏe theo tiêu ...",legal
4,56201,“Điều 21. Khám sức khỏe và điều trị bệnh nghề ...,legal


,doc_id,title,text_length,source_type
0,company_0000,Benefits And Perks,13647,company_handbook
1,company_0001,Getting Started,3147,company_handbook
2,company_0002,How We Work,6551,company_handbook
3,company_0003,Making A Career,6041,company_handbook
4,company_0004,Managing Work Devices,2937,company_handbook


# Cell 02 - Làm sạch văn bản pháp luật và handbook nội bộ

## Mục tiêu cell này
Chuẩn hóa văn bản từ hai nguồn dữ liệu:
- `legal_corpus_df`: văn bản pháp luật tiếng Việt
- `company_docs_df`: handbook nội bộ công ty 37signals

## Vì sao cần làm bước này?
Trước khi build BM25, FAISS hoặc embedding, văn bản cần được làm sạch để giảm nhiễu như:
- xuống dòng quá nhiều
- khoảng trắng thừa
- ký tự markdown/link không cần thiết
- dấu ngoặc kép hoặc format không đồng nhất

Nếu không làm sạch, retriever có thể học/truy xuất trên văn bản nhiễu, làm giảm chất lượng tìm kiếm.

## Giải thích code
Code định nghĩa hàm `clean_text()` để:
1. Chuyển dữ liệu về chuỗi văn bản
2. Xóa link markdown dạng `[text](url)` nhưng giữ lại phần `text`
3. Xóa URL thô
4. Chuẩn hóa khoảng trắng
5. Tạo thêm cột `clean_text`

Sau cell này, mỗi tài liệu sẽ có thêm bản văn bản sạch để dùng cho chunking ở bước sau.

## Output mong đợi
Bạn cần thấy:
- số dòng legal corpus không đổi
- số dòng company handbook không đổi
- preview văn bản trước và sau khi làm sạch

In [2]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = text.replace("\ufeff", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

legal_clean_df = legal_corpus_df.copy()
company_clean_df = company_docs_df.copy()

legal_clean_df["clean_text"] = legal_clean_df["context_text"].apply(clean_text)
company_clean_df["clean_text"] = company_clean_df["text"].apply(clean_text)

print("Legal clean shape:", legal_clean_df.shape)
print("Company clean shape:", company_clean_df.shape)

print("\nLegal before:")
print(legal_corpus_df.loc[0, "context_text"][:500])

print("\nLegal after:")
print(legal_clean_df.loc[0, "clean_text"][:500])

print("\nCompany before:")
print(company_docs_df.loc[0, "text"][:500])

print("\nCompany after:")
print(company_clean_df.loc[0, "clean_text"][:500])

Legal clean shape: (68663, 4)
Company clean shape: (16, 7)

Legal before:
“Điều 2. Địa vị pháp lý của Liên đoàn Luật sư Việt Nam
1. Liên đoàn Luật sư Việt Nam là tổ chức xã hội - nghề nghiệp thống nhất trong toàn quốc của các Đoàn Luật sư, các luật sư Việt Nam; có tư cách pháp nhân, có con dấu, tài khoản.
2. Biểu tượng của Liên đoàn Luật sư Việt Nam là hình tròn nền xanh da trời, chính giữa là cán cân công lý gắn với hình tượng cuốn sách, dưới cán cân công lý là dòng chữ “VIETNAM BAR FEDERATION", hai bên mỗi bên có ba dải màu vàng đậm, phía trên là ngôi sao vàng hình 

Legal after:
“Điều 2. Địa vị pháp lý của Liên đoàn Luật sư Việt Nam
1. Liên đoàn Luật sư Việt Nam là tổ chức xã hội - nghề nghiệp thống nhất trong toàn quốc của các Đoàn Luật sư, các luật sư Việt Nam; có tư cách pháp nhân, có con dấu, tài khoản.
2. Biểu tượng của Liên đoàn Luật sư Việt Nam là hình tròn nền xanh da trời, chính giữa là cán cân công lý gắn với hình tượng cuốn sách, dưới cán cân công lý là dòng chữ “VIETNAM 

# Cell 03 - Thống kê độ dài văn bản trước khi chunking

## Mục tiêu cell này
Kiểm tra độ dài văn bản sau khi làm sạch của hai nguồn dữ liệu:
- Văn bản pháp luật Việt Nam
- Handbook nội bộ công ty

## Vì sao cần làm bước này?
Chunking là bước chia tài liệu dài thành các đoạn nhỏ để RAG truy xuất chính xác hơn.  
Nếu chunk quá dài, retriever khó tìm đúng ý nhỏ.  
Nếu chunk quá ngắn, context bị mất nghĩa và LLM khó trả lời đầy đủ.

Với dataset hiện tại:
- Legal corpus vốn đã là các đoạn context pháp luật có `cid`, nên có thể giữ mỗi `cid` như một legal chunk.
- Company handbook là các file Markdown dài, nên cần chia nhỏ thành nhiều chunk.

## Giải thích code
Code sẽ:
1. Tính số ký tự của từng văn bản đã làm sạch
2. Thống kê min, max, mean, median
3. Hiển thị một số tài liệu handbook dài nhất
4. Giúp quyết định chiến lược chunking ở cell tiếp theo

## Output mong đợi
Bạn cần thấy thống kê độ dài của `legal_clean_df` và `company_clean_df`.  
Nếu handbook có file dài hàng nghìn ký tự, ta sẽ chunk handbook thành nhiều đoạn nhỏ.

In [3]:
legal_clean_df["text_chars"] = legal_clean_df["clean_text"].apply(len)
company_clean_df["text_chars"] = company_clean_df["clean_text"].apply(len)

print("Legal text length stats:")
display(legal_clean_df["text_chars"].describe())

print("Company handbook text length stats:")
display(company_clean_df["text_chars"].describe())

print("Top handbook documents by length:")
display(
    company_clean_df[["doc_id", "title", "source_path", "text_chars"]]
    .sort_values("text_chars", ascending=False)
    .head(10)
)

Legal text length stats:


count    68663.000000
mean       993.967421
std        773.543289
min          4.000000
25%        470.000000
50%        803.000000
75%       1280.000000
max      13762.000000
Name: text_chars, dtype: float64

Company handbook text length stats:


count       16.000000
mean      6183.000000
std       4688.175765
min       1060.000000
25%       2641.000000
50%       5277.500000
75%       9790.500000
max      16785.000000
Name: text_chars, dtype: float64

Top handbook documents by length:


,doc_id,title,source_path,text_chars
15,company_0015,Titles For Support,handbook-master\titles-for-support.md,16785
0,company_0000,Benefits And Perks,handbook-master\benefits-and-perks.md,13087
12,company_0012,Titles For Ops,handbook-master\titles-for-ops.md,10889
13,company_0013,Titles For Programmers,handbook-master\titles-for-programmers.md,10716
11,company_0011,Titles For Designers,handbook-master\titles-for-designers.md,9482
2,company_0002,How We Work,handbook-master\how-we-work.md,6128
14,company_0014,Titles For Qa,handbook-master\titles-for-QA.md,5569
3,company_0003,Making A Career,handbook-master\making-a-career.md,5511
10,company_0010,Statefmla,handbook-master\stateFMLA.md,5044
5,company_0005,Moonlighting,handbook-master\moonlighting.md,3486


# Cell 04 - Định nghĩa hàm chunk văn bản

## Mục tiêu cell này
Tạo hàm `chunk_text()` để chia văn bản dài thành nhiều đoạn nhỏ phục vụ RAG.

## Vì sao cần làm bước này?
Trong RAG, hệ thống không nên đưa nguyên một tài liệu dài vào retriever hoặc LLM.  
Thay vào đó, tài liệu cần được chia thành các chunk nhỏ hơn để:
- retriever tìm đúng đoạn liên quan hơn
- LLM nhận context gọn hơn
- nguồn trích dẫn rõ ràng hơn
- giảm nhiễu khi tạo câu trả lời

Với project này:
- văn bản pháp luật sẽ giữ `cid` làm `parent_id`
- handbook nội bộ sẽ giữ `doc_id` làm `parent_id`

## Giải thích code
Code định nghĩa hàm `chunk_text()`:
1. Tách văn bản theo đoạn xuống dòng
2. Gom các đoạn lại cho đến khi đạt giới hạn `max_chars`
3. Nếu đoạn quá dài, tự cắt theo kích thước ký tự
4. Thêm overlap nhỏ giữa các chunk để tránh mất ngữ cảnh ở ranh giới chunk

Sau đó code thử chunk một tài liệu handbook dài nhất để kiểm tra hàm hoạt động đúng.

## Output mong đợi
Bạn cần thấy:
- tên tài liệu được test
- độ dài gốc
- số chunk sau khi tách
- preview một vài chunk đầu tiên

In [4]:
def chunk_text(text, max_chars=1200, overlap=150):
    text = clean_text(text)
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks, current = [], ""

    for p in paragraphs:
        if len(p) > max_chars:
            if current:
                chunks.append(current.strip())
                current = ""
            start = 0
            while start < len(p):
                chunks.append(p[start:start + max_chars].strip())
                start += max_chars - overlap
        elif len(current) + len(p) + 2 <= max_chars:
            current = f"{current}\n\n{p}".strip()
        else:
            chunks.append(current.strip())
            tail = current[-overlap:] if overlap and len(current) > overlap else ""
            current = f"{tail}\n\n{p}".strip()

    if current:
        chunks.append(current.strip())

    return [c for c in chunks if len(c.strip()) > 0]

sample_row = company_clean_df.sort_values("text_chars", ascending=False).iloc[0]
sample_chunks = chunk_text(sample_row["clean_text"])

print("Tài liệu test:", sample_row["title"])
print("Độ dài gốc:", len(sample_row["clean_text"]))
print("Số chunk:", len(sample_chunks))

for i, chunk in enumerate(sample_chunks[:3]):
    print("\n--- Chunk", i, "---")
    print(chunk[:500])

Tài liệu test: Titles For Support
Độ dài gốc: 16785
Số chunk: 18

--- Chunk 0 ---
# Titles for Customer Support

--- Chunk 1 ---
| Category | Junior Customer Support Rep (L1) | Customer Support Rep (L2) | Senior Customer Support Rep (L3) | Lead Customer Support Rep (L4) | Principal Customer Support Rep (L5) |
| ----- | ----- | ----- | ----- | ----- | ----- |
| **Skill** | | | | | |
| *Technical* | Communicates basic product knowledge to customers well and with enthusiasm. | Confident in product knowledge and well versed in one product. Communicates knowledge to customers well and with enthusiasm, and able to adapt messagi

--- Chunk 2 ---
e to work independently on routine emails, and doesn't need help answering commonly asked questions. | Customer emails adhere to the style guide with little need for redirection. Able to work independently on all emails or calls. | Customer emails adhere to the style guide with no need for redirection.Able to work independently on all emails or calls,

# Cell 05 - Cải thiện hàm chunk để tránh chunk quá ngắn và cắt giữa chữ

## Mục tiêu cell này
Cải thiện hàm chunk văn bản để tạo các đoạn tài liệu chất lượng hơn cho RAG.

## Vì sao cần làm bước này?
Ở cell trước, chunk đầu tiên chỉ chứa tiêu đề và một số chunk bị cắt giữa chữ.  
Điều này không tốt cho RAG vì:
- chunk quá ngắn không đủ thông tin để trả lời
- chunk bị cắt giữa chữ làm giảm chất lượng embedding
- evidence hiển thị cho người dùng sẽ khó đọc
- LLM dễ hiểu sai nếu context bị cắt không tự nhiên

## Giải thích code
Code sẽ định nghĩa lại hàm `chunk_text()` theo hướng tốt hơn:
1. Giữ tiêu đề Markdown đi kèm nội dung phía sau
2. Cắt văn bản dài theo khoảng trắng gần nhất thay vì cắt giữa chữ
3. Dùng overlap nhỏ để không mất ngữ cảnh giữa hai chunk
4. Gộp các chunk quá ngắn với chunk lân cận nếu có thể

Sau đó code test lại trên file handbook dài nhất để kiểm tra chunk đã dễ đọc hơn chưa.

## Output mong đợi
Bạn cần thấy:
- chunk đầu tiên không còn chỉ có tiêu đề đơn lẻ
- chunk không bị bắt đầu giữa chữ quá rõ
- số chunk vẫn hợp lý

In [5]:
def split_long_block(block, max_chars=1200, overlap=120):
    parts = []
    start = 0

    while start < len(block):
        end = min(start + max_chars, len(block))

        if end < len(block):
            cut = block.rfind(" ", start + int(max_chars * 0.6), end)
            if cut != -1:
                end = cut

        parts.append(block[start:end].strip())

        if end >= len(block):
            break

        start = max(end - overlap, start + 1)

    return parts

def get_overlap_text(text, overlap=120):
    if not overlap or len(text) <= overlap:
        return ""
    tail = text[-overlap:]
    cut = tail.find(" ")
    return tail[cut + 1:].strip() if cut != -1 else tail.strip()

def chunk_text(text, max_chars=1200, overlap=120, min_chars=180):
    text = clean_text(text)
    raw_blocks = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]

    blocks = []
    pending_headings = []

    for block in raw_blocks:
        if re.match(r"^#{1,6}\s+", block):
            pending_headings.append(block)
        else:
            if pending_headings:
                block = "\n".join(pending_headings + [block])
                pending_headings = []
            blocks.append(block)

    if pending_headings and blocks:
        blocks[-1] = blocks[-1] + "\n\n" + "\n".join(pending_headings)

    chunks = []
    current = ""

    for block in blocks:
        sub_blocks = split_long_block(block, max_chars, overlap) if len(block) > max_chars else [block]

        for sub in sub_blocks:
            if len(current) + len(sub) + 2 <= max_chars:
                current = f"{current}\n\n{sub}".strip()
            else:
                if current:
                    chunks.append(current.strip())
                tail = get_overlap_text(current, overlap)
                current = f"{tail}\n\n{sub}".strip() if tail else sub

    if current:
        chunks.append(current.strip())

    merged = []
    for chunk in chunks:
        if merged and len(chunk) < min_chars and len(merged[-1]) + len(chunk) + 2 <= max_chars:
            merged[-1] = f"{merged[-1]}\n\n{chunk}".strip()
        else:
            merged.append(chunk)

    return merged

sample_row = company_clean_df.sort_values("text_chars", ascending=False).iloc[0]
sample_chunks = chunk_text(sample_row["clean_text"])

print("Tài liệu test:", sample_row["title"])
print("Độ dài gốc:", len(sample_row["clean_text"]))
print("Số chunk:", len(sample_chunks))

for i, chunk in enumerate(sample_chunks[:3]):
    print("\n--- Chunk", i, "| length:", len(chunk), "---")
    print(chunk[:700])

Tài liệu test: Titles For Support
Độ dài gốc: 16785
Số chunk: 16

--- Chunk 0 | length: 1198 ---
# Titles for Customer Support
| Category | Junior Customer Support Rep (L1) | Customer Support Rep (L2) | Senior Customer Support Rep (L3) | Lead Customer Support Rep (L4) | Principal Customer Support Rep (L5) |
| ----- | ----- | ----- | ----- | ----- | ----- |
| **Skill** | | | | | |
| *Technical* | Communicates basic product knowledge to customers well and with enthusiasm. | Confident in product knowledge and well versed in one product. Communicates knowledge to customers well and with enthusiasm, and able to adapt messaging to the situation. | Proficient in one or more products. Communicates knowledge to customers well and with enthusiasm, and able to adapt messaging to the situation. | P

--- Chunk 1 | length: 1310 ---
to work independently on routine emails, and doesn't need help answering commonly asked questions. | Customer emails

ble to work independently on routine emails, and doe

# Cell 06 - Tối ưu hàm chunk để evidence sạch và không bị lặp đoạn

## Mục tiêu cell này
Tạo phiên bản chunking ổn định hơn cho cả văn bản pháp luật và handbook nội bộ.

## Vì sao cần làm bước này?
Ở cell trước, chunk đã tốt hơn nhưng vẫn có hiện tượng lặp nội dung do overlap, đặc biệt với các bảng Markdown dài trong handbook.  
Nếu evidence bị lặp hoặc bắt đầu giữa câu quá nhiều, người đọc báo cáo/demo sẽ khó hiểu nguồn mà hệ thống truy xuất.

## Giải thích code
Code sẽ định nghĩa lại `chunk_text()` theo hướng:
1. Giữ tiêu đề Markdown đi cùng nội dung phía sau
2. Không dùng overlap mặc định để tránh lặp đoạn
3. Khi gặp block quá dài, cắt tại vị trí tự nhiên như xuống dòng, dấu chấm, dấu chấm phẩy hoặc khoảng trắng
4. Gộp chunk quá ngắn vào chunk gần nhất nếu còn đủ độ dài

Hàm này sẽ được dùng để tạo `legal_chunks` và `company_chunks` ở các cell sau.

## Output mong đợi
Chunk đầu tiên có cả tiêu đề và nội dung.  
Các chunk sau không bị lặp đoạn rõ ràng và không bắt đầu bằng chữ bị cắt giữa từ.

In [6]:
def smart_cut(text, start, max_chars):
    end = min(start + max_chars, len(text))

    if end >= len(text):
        return len(text)

    window_start = start + int(max_chars * 0.6)
    candidates = [
        text.rfind("\n", window_start, end),
        text.rfind(". ", window_start, end),
        text.rfind("; ", window_start, end),
        text.rfind(", ", window_start, end),
        text.rfind(" ", window_start, end),
    ]

    cut = max(candidates)
    return cut + 1 if cut != -1 else end

def split_long_block(block, max_chars=1200):
    parts = []
    start = 0

    while start < len(block):
        end = smart_cut(block, start, max_chars)
        part = block[start:end].strip()
        if part:
            parts.append(part)
        start = end

    return parts

def chunk_text(text, max_chars=1200, min_chars=180):
    text = clean_text(text)
    raw_blocks = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]

    blocks = []
    headings = []

    for block in raw_blocks:
        if re.match(r"^#{1,6}\s+", block):
            headings.append(block)
        else:
            if headings:
                block = "\n".join(headings + [block])
                headings = []
            blocks.append(block)

    if headings:
        if blocks:
            blocks[-1] = f"{blocks[-1]}\n\n" + "\n".join(headings)
        else:
            blocks.append("\n".join(headings))

    chunks = []
    current = ""

    for block in blocks:
        sub_blocks = split_long_block(block, max_chars) if len(block) > max_chars else [block]

        for sub in sub_blocks:
            if len(current) + len(sub) + 2 <= max_chars:
                current = f"{current}\n\n{sub}".strip()
            else:
                if current:
                    chunks.append(current.strip())
                current = sub.strip()

    if current:
        chunks.append(current.strip())

    merged = []
    for chunk in chunks:
        if merged and len(chunk) < min_chars and len(merged[-1]) + len(chunk) + 2 <= max_chars:
            merged[-1] = f"{merged[-1]}\n\n{chunk}".strip()
        else:
            merged.append(chunk)

    return merged

sample_row = company_clean_df.sort_values("text_chars", ascending=False).iloc[0]
sample_chunks = chunk_text(sample_row["clean_text"])

print("Tài liệu test:", sample_row["title"])
print("Độ dài gốc:", len(sample_row["clean_text"]))
print("Số chunk:", len(sample_chunks))

for i, chunk in enumerate(sample_chunks[:3]):
    print("\n--- Chunk", i, "| length:", len(chunk), "---")
    print(chunk[:700])

Tài liệu test: Titles For Support
Độ dài gốc: 16785
Số chunk: 15

--- Chunk 0 | length: 1198 ---
# Titles for Customer Support
| Category | Junior Customer Support Rep (L1) | Customer Support Rep (L2) | Senior Customer Support Rep (L3) | Lead Customer Support Rep (L4) | Principal Customer Support Rep (L5) |
| ----- | ----- | ----- | ----- | ----- | ----- |
| **Skill** | | | | | |
| *Technical* | Communicates basic product knowledge to customers well and with enthusiasm. | Confident in product knowledge and well versed in one product. Communicates knowledge to customers well and with enthusiasm, and able to adapt messaging to the situation. | Proficient in one or more products. Communicates knowledge to customers well and with enthusiasm, and able to adapt messaging to the situation. | P

--- Chunk 1 | length: 1196 ---
adhere to the style guide with little need for redirection. Able to work independently on all emails or calls. | Customer emails adhere to the style guide with no need fo

# Cell 07 - Tạo legal chunks từ corpus pháp luật Việt Nam

## Mục tiêu cell này
Tạo file `legal_chunks.csv` từ `legal_clean_df`.

## Vì sao cần làm bước này?
`legal_corpus_df` hiện có 68,663 đoạn pháp luật, mỗi đoạn có một `cid`.  
Trong RAG, mỗi đoạn này sẽ trở thành một hoặc nhiều chunk để retriever tìm kiếm.

Với đoạn ngắn hoặc vừa, hệ thống giữ nguyên thành 1 chunk.  
Với đoạn quá dài, hệ thống chia nhỏ thành nhiều chunk nhưng vẫn giữ `parent_id = cid`.

Điều này rất quan trọng vì sau này khi đánh giá retrieval:
- retriever trả về `chunk_id`
- ta quy về `parent_id/cid`
- so sánh với `relevant_cids` trong validation/test

## Giải thích code
Code sẽ:
1. Duyệt từng dòng trong `legal_clean_df`
2. Dùng `chunk_text()` để chia `clean_text`
3. Tạo các cột:
   - `chunk_id`: mã chunk duy nhất
   - `parent_id`: cid gốc của đoạn pháp luật
   - `source_type`: legal
   - `chunk_index`: thứ tự chunk trong cùng cid
   - `chunk_text`: nội dung chunk
   - `language`: Vietnamese
4. Lưu kết quả vào `data/chunks/legal_chunks.csv`

## Output mong đợi
Bạn cần thấy số lượng legal chunks lớn hơn hoặc bằng 68,663.  
Nếu một số văn bản dài bị chia nhỏ, tổng số chunk sẽ lớn hơn số cid gốc.

In [7]:
legal_chunk_rows = []

for _, row in legal_clean_df.iterrows():
    cid = int(row["cid"])
    chunks = chunk_text(row["clean_text"], max_chars=1200, min_chars=80)

    for idx, chunk in enumerate(chunks):
        legal_chunk_rows.append({
            "chunk_id": f"legal_{cid}_{idx:03d}",
            "parent_id": cid,
            "source_type": "legal",
            "title": f"legal_cid_{cid}",
            "chunk_index": idx,
            "chunk_text": chunk,
            "chunk_chars": len(chunk),
            "language": "Vietnamese"
        })

legal_chunks_df = pd.DataFrame(legal_chunk_rows)

out_path = CHUNK_DIR / "legal_chunks.csv"
legal_chunks_df.to_csv(out_path, index=False, encoding="utf-8-sig")

print("Legal chunks shape:", legal_chunks_df.shape)
print("Unique parent cid:", legal_chunks_df["parent_id"].nunique())
print("Đã lưu:", out_path)

display(legal_chunks_df["chunk_chars"].describe())
display(legal_chunks_df.head())

Legal chunks shape: (92898, 8)
Unique parent cid: 68663
Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\data\chunks\legal_chunks.csv


count    92898.000000
mean       734.401214
std        383.624630
min          3.000000
25%        396.000000
50%        730.000000
75%       1194.000000
max       1200.000000
Name: chunk_chars, dtype: float64

,chunk_id,parent_id,source_type,title,chunk_index,chunk_text,chunk_chars,language
0,legal_142820_000,142820,legal,legal_cid_142820,0,“Điều 2. Địa vị pháp lý của Liên đoàn Luật sư ...,767,Vietnamese
1,legal_27817_000,27817,legal,legal_cid_27817,0,"""Điều 7. Tên hợp tác xã, liên hiệp hợp tác xã\...",1194,Vietnamese
2,legal_27817_001,27817,legal,legal_cid_27817,1,khích và tạo điều kiện thuận lợi cho các hợp t...,212,Vietnamese
3,legal_72117_000,72117,legal,legal_cid_72117,0,"Cơ quan đăng ký hợp tác xã\n1. Khi thành lập, ...",811,Vietnamese
4,legal_33215_000,33215,legal,legal_cid_33215,0,"""1. Sử dụng lái xe bảo đảm sức khỏe theo tiêu ...",394,Vietnamese


# Cell 08 - Kiểm tra các legal chunks quá ngắn

## Mục tiêu cell này
Kiểm tra các chunk pháp luật có độ dài quá ngắn để xem chúng có phải dữ liệu nhiễu hay không.

## Vì sao cần làm bước này?
Trong RAG, chunk quá ngắn thường không có đủ thông tin để truy xuất hoặc sinh câu trả lời.  
Ví dụ chunk chỉ có vài ký tự như dấu câu, số thứ tự hoặc chữ bị tách rời sẽ làm giảm chất lượng BM25, embedding và reranker.

Tuy nhiên, ta không xóa ngay vì cần đảm bảo không làm mất `parent_id/cid` quan trọng dùng cho đánh giá retrieval.

## Giải thích code
Code sẽ:
1. Lọc các chunk có độ dài dưới 50 ký tự
2. Đếm số chunk quá ngắn
3. Kiểm tra các `parent_id` chỉ có toàn chunk ngắn hay không
4. Hiển thị vài ví dụ để quyết định có nên lọc ở cell tiếp theo

## Output mong đợi
Bạn cần thấy:
- số lượng chunk dưới 50 ký tự
- số `parent_id` bị ảnh hưởng
- một số ví dụ chunk quá ngắn

In [8]:
SHORT_CHUNK_THRESHOLD = 50

short_chunks_df = legal_chunks_df[legal_chunks_df["chunk_chars"] < SHORT_CHUNK_THRESHOLD].copy()

parent_chunk_counts = legal_chunks_df.groupby("parent_id")["chunk_id"].count()
short_parent_counts = short_chunks_df.groupby("parent_id")["chunk_id"].count()

only_short_parent_ids = [
    pid for pid, n_short in short_parent_counts.items()
    if parent_chunk_counts.loc[pid] == n_short
]

print("Ngưỡng chunk ngắn:", SHORT_CHUNK_THRESHOLD)
print("Số legal chunks quá ngắn:", len(short_chunks_df))
print("Số parent_id có chunk quá ngắn:", short_chunks_df["parent_id"].nunique())
print("Số parent_id chỉ gồm toàn chunk ngắn:", len(only_short_parent_ids))

display(
    short_chunks_df[["chunk_id", "parent_id", "chunk_index", "chunk_chars", "chunk_text"]]
    .sort_values("chunk_chars")
    .head(20)
)

Ngưỡng chunk ngắn: 50
Số legal chunks quá ngắn: 1573
Số parent_id có chunk quá ngắn: 1573
Số parent_id chỉ gồm toàn chunk ngắn: 28


,chunk_id,parent_id,chunk_index,chunk_chars,chunk_text
2801,legal_164514_001,164514,1,3,...
62719,legal_75534_001,75534,1,3,vệ.
19689,legal_211101_001,211101,1,3,...
5835,legal_10863_001,10863,1,3,án.
46000,legal_115737_001,115737,1,3,...
45235,legal_91634_002,91634,2,3,bỏ.
80770,legal_52768_002,52768,2,3,vụ.
41099,legal_223054_001,223054,1,3,...
53575,legal_77364_002,77364,2,3,nổ.
80528,legal_127777_001,127777,1,3,...


# Cell 09 - Lọc legal chunks quá ngắn nhưng vẫn giữ coverage cid

## Mục tiêu cell này
Tạo phiên bản `legal_chunks_clean.csv` sau khi loại các chunk quá ngắn không hữu ích cho RAG.

## Vì sao cần làm bước này?
Các chunk quá ngắn như `...`, `vệ.`, `án.` không đủ ngữ nghĩa để BM25, embedding hoặc reranker truy xuất chính xác.  
Nếu giữ lại, chúng có thể làm nhiễu index và làm giảm chất lượng retrieval.

Tuy nhiên, không được xóa bừa tất cả chunk ngắn vì một số `parent_id/cid` chỉ có chunk ngắn.  
Nếu xóa hết các `cid` này, ground truth trong train/validation/test có thể không còn tồn tại trong corpus.

## Giải thích code
Code sẽ:
1. Giữ toàn bộ chunk có độ dài từ 50 ký tự trở lên
2. Với `parent_id` chỉ có toàn chunk ngắn, giữ lại chunk dài nhất để không mất coverage
3. Lưu chunk bị loại vào report
4. Lưu corpus sạch vào `data/chunks/legal_chunks_clean.csv`
5. Kiểm tra số `parent_id` còn lại có vẫn bằng số `cid` gốc hay không

## Output mong đợi
Bạn cần thấy:
- số chunk giảm xuống
- số `parent_id` vẫn giữ nguyên là 68,663
- không mất coverage cid

In [9]:
SHORT_CHUNK_THRESHOLD = 50

parent_chunk_counts = legal_chunks_df.groupby("parent_id")["chunk_id"].count()
short_chunks_df = legal_chunks_df[legal_chunks_df["chunk_chars"] < SHORT_CHUNK_THRESHOLD].copy()
short_parent_counts = short_chunks_df.groupby("parent_id")["chunk_id"].count()

only_short_parent_ids = [
    pid for pid, n_short in short_parent_counts.items()
    if parent_chunk_counts.loc[pid] == n_short
]

long_mask = legal_chunks_df["chunk_chars"] >= SHORT_CHUNK_THRESHOLD

keep_short_idx = (
    legal_chunks_df[legal_chunks_df["parent_id"].isin(only_short_parent_ids)]
    .sort_values(["parent_id", "chunk_chars"], ascending=[True, False])
    .groupby("parent_id")
    .head(1)
    .index
)

keep_mask = long_mask | legal_chunks_df.index.isin(keep_short_idx)

legal_chunks_clean_df = legal_chunks_df[keep_mask].copy().reset_index(drop=True)
removed_legal_chunks_df = legal_chunks_df[~keep_mask].copy().reset_index(drop=True)

clean_path = CHUNK_DIR / "legal_chunks_clean.csv"
removed_path = CHUNK_DIR / "removed_short_legal_chunks.csv"

legal_chunks_clean_df.to_csv(clean_path, index=False, encoding="utf-8-sig")
removed_legal_chunks_df.to_csv(removed_path, index=False, encoding="utf-8-sig")

print("Legal chunks ban đầu:", len(legal_chunks_df))
print("Legal chunks sau lọc:", len(legal_chunks_clean_df))
print("Số chunk bị loại:", len(removed_legal_chunks_df))
print("Unique parent_id ban đầu:", legal_chunks_df["parent_id"].nunique())
print("Unique parent_id sau lọc:", legal_chunks_clean_df["parent_id"].nunique())
print("Đã lưu corpus sạch:", clean_path)
print("Đã lưu chunk bị loại:", removed_path)

display(legal_chunks_clean_df["chunk_chars"].describe())
display(removed_legal_chunks_df[["chunk_id", "parent_id", "chunk_chars", "chunk_text"]].head(10))

Legal chunks ban đầu: 92898
Legal chunks sau lọc: 91353
Số chunk bị loại: 1545
Unique parent_id ban đầu: 68663
Unique parent_id sau lọc: 68663
Đã lưu corpus sạch: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\data\chunks\legal_chunks_clean.csv
Đã lưu chunk bị loại: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\data\chunks\removed_short_legal_chunks.csv


count    91353.000000
mean       746.388635
std        375.517356
min          4.000000
25%        412.000000
50%        743.000000
75%       1195.000000
max       1200.000000
Name: chunk_chars, dtype: float64

,chunk_id,parent_id,chunk_chars,chunk_text
0,legal_48911_002,48911,19,đăng ký kinh doanh.
1,legal_133068_001,133068,11,vi quản lý.
2,legal_72918_001,72918,6,"quan."""
3,legal_112955_001,112955,20,phương thức quốc tế.
4,legal_179361_001,179361,36,theo yêu cầu của hồ sơ mời thầu.\n...
5,legal_61490_001,61490,40,"36, 38 và khoản 1 Điều 39 của Luật này.”"
6,legal_146772_001,146772,8,bị chết.
7,legal_132436_001,132436,5,giải.
8,legal_61694_003,61694,6,tháng.
9,legal_70771_001,70771,20,lưu của mỗi cơ quan.


# Cell 10 - Kiểm tra coverage của legal chunks sau khi lọc

## Mục tiêu cell này
Đảm bảo sau khi lọc chunk quá ngắn, toàn bộ `relevant_cids` trong train, validation và test vẫn còn tồn tại trong `legal_chunks_clean.csv`.

## Vì sao cần làm bước này?
Ở cell trước, ta đã xóa một số legal chunks quá ngắn để làm sạch corpus cho RAG.  
Tuy nhiên, nếu xóa nhầm toàn bộ chunk của một `cid`, thì retriever sẽ không bao giờ tìm được tài liệu đúng cho các câu hỏi có `relevant_cids` đó.

Vì vậy, bước này kiểm tra lại coverage:
- train_strict
- val_strict
- test_final

## Giải thích code
Code sẽ:
1. Lấy toàn bộ `parent_id` còn lại trong `legal_chunks_clean_df`
2. Lấy toàn bộ `relevant_cids` của từng split
3. Kiểm tra có `cid` nào bị thiếu khỏi legal chunks sạch không
4. Lưu báo cáo vào `data/chunks/legal_chunk_coverage_report.csv`

## Output mong đợi
`missing_cids` của train, validation và test đều phải bằng 0.  
`coverage_rate` phải bằng 1.0 cho cả ba split.

In [10]:
clean_parent_ids = set(legal_chunks_clean_df["parent_id"].astype(int))

def get_cid_set(df):
    return set(cid for cids in df["relevant_cids"] for cid in cids)

def chunk_coverage_report(split_name, df):
    relevant_cids = get_cid_set(df)
    missing_cids = sorted(relevant_cids - clean_parent_ids)
    return {
        "split": split_name,
        "unique_relevant_cids": len(relevant_cids),
        "missing_cids": len(missing_cids),
        "coverage_rate": round((len(relevant_cids) - len(missing_cids)) / len(relevant_cids), 6),
        "missing_examples": missing_cids[:10]
    }

chunk_coverage_df = pd.DataFrame([
    chunk_coverage_report("train_strict", train_df),
    chunk_coverage_report("val_strict", val_df),
    chunk_coverage_report("test_final", test_df)
])

coverage_path = CHUNK_DIR / "legal_chunk_coverage_report.csv"
chunk_coverage_df.to_csv(coverage_path, index=False, encoding="utf-8-sig")

print("Đã lưu:", coverage_path)
display(chunk_coverage_df)

Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\data\chunks\legal_chunk_coverage_report.csv


,split,unique_relevant_cids,missing_cids,coverage_rate,missing_examples
0,train_strict,51431,0,1.0,[]
1,val_strict,8109,0,1.0,[]
2,test_final,23907,0,1.0,[]


# Cell 11 - Tạo chunks cho handbook nội bộ công ty

## Mục tiêu cell này
Tạo file `company_handbook_chunks.csv` từ bộ tài liệu nội bộ 37signals Employee Handbook.

## Vì sao cần làm bước này?
Handbook nội bộ công ty hiện chỉ có 16 file Markdown gốc, nhưng nhiều file khá dài, có file hơn 16,000 ký tự.  
Nếu đưa nguyên một file dài vào retriever, hệ thống có thể truy xuất không chính xác vì một file chứa nhiều chủ đề khác nhau như benefits, thiết bị, cách làm việc, hệ thống nội bộ, vai trò nhân viên.

Vì vậy, ta cần chia mỗi file handbook thành nhiều chunk nhỏ hơn.  
Mỗi chunk sẽ là một đơn vị truy xuất trong RAG.

## Giải thích code
Code sẽ:
1. Duyệt từng tài liệu trong `company_clean_df`
2. Dùng hàm `chunk_text()` đã tối ưu ở các cell trước
3. Tạo các trường metadata:
   - `chunk_id`: mã chunk nội bộ duy nhất
   - `parent_id`: doc_id gốc của file handbook
   - `source_type`: company_handbook
   - `title`: tiêu đề tài liệu gốc
   - `source_path`: đường dẫn file Markdown gốc
   - `chunk_index`: thứ tự chunk trong tài liệu
   - `chunk_text`: nội dung chunk
   - `language`: English
4. Lưu kết quả vào `data/chunks/company_handbook_chunks.csv`

## Output mong đợi
Bạn cần thấy số lượng company chunks lớn hơn 16, vì 16 file handbook gốc đã được chia thành nhiều đoạn nhỏ hơn để phục vụ RAG.

In [11]:
company_chunk_rows = []

for _, row in company_clean_df.iterrows():
    doc_id = row["doc_id"]
    chunks = chunk_text(row["clean_text"], max_chars=1200, min_chars=120)

    for idx, chunk in enumerate(chunks):
        company_chunk_rows.append({
            "chunk_id": f"{doc_id}_{idx:03d}",
            "parent_id": doc_id,
            "source_type": "company_handbook",
            "title": row["title"],
            "source_path": row["source_path"],
            "chunk_index": idx,
            "chunk_text": chunk,
            "chunk_chars": len(chunk),
            "language": "English"
        })

company_chunks_df = pd.DataFrame(company_chunk_rows)

out_path = CHUNK_DIR / "company_handbook_chunks.csv"
company_chunks_df.to_csv(out_path, index=False, encoding="utf-8-sig")

print("Company documents gốc:", len(company_clean_df))
print("Company chunks:", len(company_chunks_df))
print("Unique parent_id:", company_chunks_df["parent_id"].nunique())
print("Đã lưu:", out_path)

display(company_chunks_df["chunk_chars"].describe())
display(company_chunks_df.head())

Company documents gốc: 16
Company chunks: 96
Unique parent_id: 16
Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\data\chunks\company_handbook_chunks.csv


count      96.000000
mean     1028.489583
std       261.925542
min        35.000000
25%       981.250000
50%      1164.000000
75%      1196.000000
max      1199.000000
Name: chunk_chars, dtype: float64

,chunk_id,parent_id,source_type,title,source_path,chunk_index,chunk_text,chunk_chars,language
0,company_0000_000,company_0000,company_handbook,Benefits And Perks,handbook-master\benefits-and-perks.md,0,# Benefits & Perks\n## Health Insurance\nDetai...,963,English
1,company_0000_001,company_0000,company_handbook,Benefits And Perks,handbook-master\benefits-and-perks.md,1,### Dental Insurance\nDental insurance in the ...,1016,English
2,company_0000_002,company_0000,company_handbook,Benefits And Perks,handbook-master\benefits-and-perks.md,2,### Health Insurance for Staff Outside the US\...,776,English
3,company_0000_003,company_0000,company_handbook,Benefits And Perks,handbook-master\benefits-and-perks.md,3,### Disability Insurance\n37signals provides M...,883,English
4,company_0000_004,company_0000,company_handbook,Benefits And Perks,handbook-master\benefits-and-perks.md,4,"If you’re outside the US, 37signals will match...",1056,English


# Cell 12 - Kiểm tra company handbook chunks quá ngắn

## Mục tiêu cell này
Kiểm tra các chunk handbook nội bộ có độ dài quá ngắn để quyết định có cần lọc hay không.

## Vì sao cần làm bước này?
Một số chunk quá ngắn có thể chỉ là tiêu đề hoặc đoạn markdown ít thông tin.  
Nếu đưa các chunk này vào BM25/FAISS, chúng có thể làm nhiễu quá trình truy xuất.

Tuy nhiên, khác với legal corpus, handbook nội bộ không có ground truth `cid`, nên việc lọc chunk ngắn chủ yếu nhằm tăng chất lượng truy vấn demo.

## Giải thích code
Code sẽ:
1. Lọc các company chunks có độ dài dưới 120 ký tự
2. Đếm số lượng chunk ngắn
3. Hiển thị nội dung các chunk ngắn để kiểm tra thủ công
4. Chưa xóa dữ liệu ở cell này

## Output mong đợi
Bạn cần thấy số chunk handbook quá ngắn và nội dung của chúng.

In [12]:
SHORT_COMPANY_THRESHOLD = 120

short_company_chunks_df = company_chunks_df[
    company_chunks_df["chunk_chars"] < SHORT_COMPANY_THRESHOLD
].copy()

print("Ngưỡng chunk ngắn:", SHORT_COMPANY_THRESHOLD)
print("Số company chunks quá ngắn:", len(short_company_chunks_df))
print("Số parent_id bị ảnh hưởng:", short_company_chunks_df["parent_id"].nunique())

display(
    short_company_chunks_df[
        ["chunk_id", "parent_id", "title", "chunk_index", "chunk_chars", "chunk_text"]
    ].sort_values("chunk_chars")
)

Ngưỡng chunk ngắn: 120
Số company chunks quá ngắn: 1
Số parent_id bị ảnh hưởng: 1


,chunk_id,parent_id,title,chunk_index,chunk_chars,chunk_text
95,company_0015_014,company_0015,Titles For Support,14,35,making and professional guidance. |


# Cell 13 - Lọc company handbook chunks quá ngắn

## Mục tiêu cell này
Loại các chunk handbook nội bộ quá ngắn và ít giá trị ngữ nghĩa.

## Vì sao cần làm bước này?
Chunk quá ngắn như `making and professional guidance. |` không đủ thông tin để trả lời câu hỏi.  
Nếu giữ lại, nó có thể làm nhiễu BM25, FAISS hoặc reranker.

Với company handbook, ta có thể lọc chunk ngắn an toàn hơn legal corpus vì handbook không dùng `cid` làm ground truth đánh giá retrieval.

## Giải thích code
Code sẽ:
1. Giữ lại các company chunks có độ dài từ 120 ký tự trở lên
2. Lưu chunk bị loại vào `removed_short_company_chunks.csv`
3. Lưu corpus handbook sạch vào `company_handbook_chunks_clean.csv`
4. Kiểm tra số tài liệu gốc `parent_id` còn lại

## Output mong đợi
Bạn cần thấy:
- Company chunks ban đầu: 96
- Company chunks sau lọc: 95
- Số chunk bị loại: 1
- Unique parent_id vẫn là 16

In [13]:
SHORT_COMPANY_THRESHOLD = 120

company_chunks_clean_df = company_chunks_df[
    company_chunks_df["chunk_chars"] >= SHORT_COMPANY_THRESHOLD
].copy().reset_index(drop=True)

removed_company_chunks_df = company_chunks_df[
    company_chunks_df["chunk_chars"] < SHORT_COMPANY_THRESHOLD
].copy().reset_index(drop=True)

clean_path = CHUNK_DIR / "company_handbook_chunks_clean.csv"
removed_path = CHUNK_DIR / "removed_short_company_chunks.csv"

company_chunks_clean_df.to_csv(clean_path, index=False, encoding="utf-8-sig")
removed_company_chunks_df.to_csv(removed_path, index=False, encoding="utf-8-sig")

print("Company chunks ban đầu:", len(company_chunks_df))
print("Company chunks sau lọc:", len(company_chunks_clean_df))
print("Số chunk bị loại:", len(removed_company_chunks_df))
print("Unique parent_id sau lọc:", company_chunks_clean_df["parent_id"].nunique())
print("Đã lưu corpus sạch:", clean_path)
print("Đã lưu chunk bị loại:", removed_path)

display(company_chunks_clean_df["chunk_chars"].describe())
display(removed_company_chunks_df[["chunk_id", "title", "chunk_chars", "chunk_text"]])

Company chunks ban đầu: 96
Company chunks sau lọc: 95
Số chunk bị loại: 1
Unique parent_id sau lọc: 16
Đã lưu corpus sạch: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\data\chunks\company_handbook_chunks_clean.csv
Đã lưu chunk bị loại: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\data\chunks\removed_short_company_chunks.csv


count      95.000000
mean     1038.947368
std       242.330501
min       128.000000
25%       987.000000
50%      1165.000000
75%      1196.000000
max      1199.000000
Name: chunk_chars, dtype: float64

,chunk_id,title,chunk_chars,chunk_text
0,company_0015_014,Titles For Support,35,making and professional guidance. |


# Cell 14 - Gộp legal chunks và company handbook chunks thành corpus chung

## Mục tiêu cell này
Tạo một file corpus thống nhất gồm cả:
- các chunk pháp luật Việt Nam
- các chunk handbook nội bộ công ty

## Vì sao cần làm bước này?
Ứng dụng RAG của project cần truy xuất từ hai nguồn tri thức:
1. `legal`: quy định pháp luật Việt Nam
2. `company_handbook`: chính sách nội bộ công ty

Nếu lưu hai nguồn riêng lẻ, các bước build BM25, FAISS, reranker và demo Streamlit sẽ phức tạp hơn.  
Vì vậy, ta chuẩn hóa chúng về cùng một bảng `all_chunks.csv`.

## Giải thích code
Code sẽ:
1. Chuẩn hóa cột của `legal_chunks_clean_df` và `company_chunks_clean_df`
2. Thêm `source_path` rỗng cho legal chunks vì legal corpus không có file Markdown gốc
3. Tạo `corpus_id` duy nhất cho mỗi chunk
4. Tạo cột `retrieval_text` bằng cách ghép `title` và `chunk_text`
5. Lưu corpus chung vào `data/chunks/all_chunks.csv`

Cột `retrieval_text` sẽ là văn bản chính dùng để build BM25 và embedding ở các notebook sau.

## Output mong đợi
Bạn cần thấy tổng số chunks khoảng:
- 91,353 legal chunks
- 95 company handbook chunks
- tổng khoảng 91,448 chunks

In [14]:
common_cols = [
    "chunk_id",
    "parent_id",
    "source_type",
    "title",
    "source_path",
    "chunk_index",
    "chunk_text",
    "chunk_chars",
    "language"
]

legal_ready_df = legal_chunks_clean_df.copy()
company_ready_df = company_chunks_clean_df.copy()

legal_ready_df["source_path"] = ""

legal_ready_df = legal_ready_df[common_cols]
company_ready_df = company_ready_df[common_cols]

all_chunks_df = pd.concat([legal_ready_df, company_ready_df], ignore_index=True)

all_chunks_df["corpus_id"] = [f"chunk_{i:06d}" for i in range(len(all_chunks_df))]
all_chunks_df["retrieval_text"] = (
    all_chunks_df["title"].fillna("").astype(str)
    + "\n"
    + all_chunks_df["chunk_text"].fillna("").astype(str)
)

all_chunks_df = all_chunks_df[
    ["corpus_id"] + common_cols + ["retrieval_text"]
]

out_path = CHUNK_DIR / "all_chunks.csv"
all_chunks_df.to_csv(out_path, index=False, encoding="utf-8-sig")

print("All chunks shape:", all_chunks_df.shape)
print("Đã lưu:", out_path)

display(
    all_chunks_df.groupby("source_type")
    .agg(
        chunks=("chunk_id", "count"),
        unique_parent=("parent_id", "nunique"),
        avg_chars=("chunk_chars", "mean"),
        min_chars=("chunk_chars", "min"),
        max_chars=("chunk_chars", "max")
    )
    .reset_index()
)

display(all_chunks_df.head())

All chunks shape: (91448, 11)
Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\data\chunks\all_chunks.csv


,source_type,chunks,unique_parent,avg_chars,min_chars,max_chars
0,company_handbook,95,16,1038.947368,128,1199
1,legal,91353,68663,746.388635,4,1200


,corpus_id,chunk_id,parent_id,source_type,title,source_path,chunk_index,chunk_text,chunk_chars,language,retrieval_text
0,chunk_000000,legal_142820_000,142820,legal,legal_cid_142820,,0,“Điều 2. Địa vị pháp lý của Liên đoàn Luật sư ...,767,Vietnamese,legal_cid_142820\n“Điều 2. Địa vị pháp lý của ...
1,chunk_000001,legal_27817_000,27817,legal,legal_cid_27817,,0,"""Điều 7. Tên hợp tác xã, liên hiệp hợp tác xã\...",1194,Vietnamese,"legal_cid_27817\n""Điều 7. Tên hợp tác xã, liên..."
2,chunk_000002,legal_27817_001,27817,legal,legal_cid_27817,,1,khích và tạo điều kiện thuận lợi cho các hợp t...,212,Vietnamese,legal_cid_27817\nkhích và tạo điều kiện thuận ...
3,chunk_000003,legal_72117_000,72117,legal,legal_cid_72117,,0,"Cơ quan đăng ký hợp tác xã\n1. Khi thành lập, ...",811,Vietnamese,legal_cid_72117\nCơ quan đăng ký hợp tác xã\n1...
4,chunk_000004,legal_33215_000,33215,legal,legal_cid_33215,,0,"""1. Sử dụng lái xe bảo đảm sức khỏe theo tiêu ...",394,Vietnamese,"legal_cid_33215\n""1. Sử dụng lái xe bảo đảm sứ..."


# Cell 15 - Tạo báo cáo tổng kết chunking

## Mục tiêu cell này
Tổng hợp kết quả chunking của toàn bộ corpus RAG.

## Vì sao cần làm bước này?
Sau khi tạo chunks, ta cần lưu lại các thống kê quan trọng để:
- biết corpus cuối cùng có bao nhiêu chunk
- kiểm tra số nguồn dữ liệu legal/company
- kiểm tra độ dài chunk có hợp lý không
- dùng bảng này trong báo cáo hoặc thuyết trình đồ án

Đây là bước chốt của notebook `03_preprocessing_and_chunking.ipynb`.

## Giải thích code
Code sẽ tạo bảng thống kê theo `source_type`, gồm:
- số lượng chunk
- số lượng tài liệu gốc
- độ dài trung bình
- độ dài nhỏ nhất
- độ dài lớn nhất
- ngôn ngữ

Sau đó lưu kết quả vào:
`data/chunks/chunk_summary.csv`

## Output mong đợi
Bạn cần thấy 2 dòng:
- `legal`
- `company_handbook`

và file `chunk_summary.csv` được lưu thành công.

In [15]:
chunk_summary_df = (
    all_chunks_df
    .groupby("source_type")
    .agg(
        num_chunks=("chunk_id", "count"),
        num_parent_docs=("parent_id", "nunique"),
        avg_chunk_chars=("chunk_chars", "mean"),
        min_chunk_chars=("chunk_chars", "min"),
        max_chunk_chars=("chunk_chars", "max"),
        languages=("language", lambda x: ", ".join(sorted(set(x))))
    )
    .reset_index()
)

chunk_summary_df["avg_chunk_chars"] = chunk_summary_df["avg_chunk_chars"].round(2)

summary_path = CHUNK_DIR / "chunk_summary.csv"
chunk_summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("Đã lưu:", summary_path)
display(chunk_summary_df)

Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\data\chunks\chunk_summary.csv


,source_type,num_chunks,num_parent_docs,avg_chunk_chars,min_chunk_chars,max_chunk_chars,languages
0,company_handbook,95,16,1038.95,128,1199,English
1,legal,91353,68663,746.39,4,1200,Vietnamese


# Cell 16 - Kiểm tra cuối các file chunk đã tạo

## Mục tiêu cell này
Kiểm tra toàn bộ file đầu ra của notebook `03_preprocessing_and_chunking.ipynb`.

## Vì sao cần làm bước này?
Các notebook sau sẽ dùng trực tiếp các file chunk để build:
- BM25 index
- FAISS dense index
- Hybrid retrieval
- Reranker RAG
- Corrective RAG
- Streamlit demo

Nếu thiếu file ở bước này, các notebook sau sẽ lỗi hoặc dùng nhầm dữ liệu.

## Giải thích code
Code sẽ kiểm tra sự tồn tại và dung lượng của các file quan trọng:
- `legal_chunks_clean.csv`
- `company_handbook_chunks_clean.csv`
- `all_chunks.csv`
- `chunk_summary.csv`
- `legal_chunk_coverage_report.csv`

## Output mong đợi
Tất cả file cần có trạng thái `OK`.

In [16]:
required_chunk_files = [
    CHUNK_DIR / "legal_chunks_clean.csv",
    CHUNK_DIR / "company_handbook_chunks_clean.csv",
    CHUNK_DIR / "all_chunks.csv",
    CHUNK_DIR / "chunk_summary.csv",
    CHUNK_DIR / "legal_chunk_coverage_report.csv"
]

chunk_check_df = pd.DataFrame([
    {
        "file": str(path.relative_to(PROJECT)),
        "status": "OK" if path.exists() else "MISSING",
        "size_mb": round(path.stat().st_size / 1024 / 1024, 2) if path.exists() else 0
    }
    for path in required_chunk_files
])

display(chunk_check_df)

print("Tổng file OK:", (chunk_check_df["status"] == "OK").sum(), "/", len(chunk_check_df))
print("Corpus cuối cùng:", len(all_chunks_df), "chunks")
print("Nguồn dữ liệu:", all_chunks_df["source_type"].value_counts().to_dict())

,file,status,size_mb
0,data\chunks\legal_chunks_clean.csv,OK,92.66
1,data\chunks\company_handbook_chunks_clean.csv,OK,0.11
2,data\chunks\all_chunks.csv,OK,182.62
3,data\chunks\chunk_summary.csv,OK,0.00
4,data\chunks\legal_chunk_coverage_report.csv,OK,0.00


Tổng file OK: 5 / 5
Corpus cuối cùng: 91448 chunks
Nguồn dữ liệu: {'legal': 91353, 'company_handbook': 95}



Hãy tưởng tượng app của mình là chatbot nội bộ cho HR/pháp chế:

```text
Nhân viên hỏi:
"Nếu tôi nghỉ việc thì công ty còn trả bảo hiểm/phúc lợi đến khi nào?"
```

Nếu chưa có file 03, hệ thống chỉ có nguyên file dài:

```text
benefits-and-perks.md dài hơn 13,000 ký tự
severance.md dài hơn 1,000 ký tự
legal documents rất nhiều đoạn dài/ngắn lẫn lộn
```

Khi đó retriever có thể lấy nguyên một file dài, LLM đọc rất nhiều phần không liên quan, câu trả lời dễ lan man.

Sau file 03, tài liệu được biến thành nhiều mảnh nhỏ:

```text
Benefits And Perks - chunk 0: Health Insurance
Benefits And Perks - chunk 1: Dental Insurance
Benefits And Perks - chunk 2: Health Insurance for Staff Outside the US
Benefits And Perks - chunk 3: Disability Insurance
...
Severance - chunk 0: Severance policy
```

Lúc nhân viên hỏi về phúc lợi khi nghỉ việc, RAG chỉ cần lấy đúng chunk liên quan, ví dụ:

```text
chunk từ benefits-and-perks.md nói về coverage khi resign/terminated
chunk từ severance.md nói về severance package
chunk pháp luật Việt Nam liên quan nếu cần đối chiếu
```

Rồi app trả lời kiểu:

```text
Theo handbook nội bộ, quyền lợi bảo hiểm có thể kết thúc vào ngày cuối cùng của tháng mà nhân viên nghỉ việc. Nếu áp dụng cho nhân viên Việt Nam, HR cần đối chiếu thêm quy định pháp luật Việt Nam và hợp đồng lao động cụ thể.

Nguồn:
1. Benefits And Perks - chunk ...
2. Severance - chunk ...
3. Legal cid ...
```

Nói ngắn gọn: **file 03 giống bước cắt tài liệu công ty và luật thành các “mảnh bằng chứng” nhỏ, sạch, dễ tìm**.

Ví dụ khác:

```text
Câu hỏi:
"Công ty có chính sách gì về thiết bị làm việc khi nghỉ việc?"
```

Nếu chưa chunk:

```text
Retriever phải tìm trong nguyên file managing-work-devices.md
```

Sau khi chunk:

```text
Retriever tìm đúng chunk nói về work devices, laptop, thiết bị công ty
```

App sẽ trả lời có nguồn:

```text
Theo handbook nội bộ, thiết bị làm việc do công ty cấp cần được quản lý và hoàn trả khi không còn phục vụ công việc.

Nguồn:
Managing Work Devices - chunk ...
```

Với phần pháp luật Việt Nam cũng tương tự. Ví dụ câu hỏi:

```text
"Lương thử việc được quy định thế nào?"
```

Dataset pháp luật có `relevant_cids`. File 03 biến mỗi `cid` thành một hoặc nhiều chunk nhỏ. Khi retriever tìm được chunk có `parent_id = cid đúng`, mình tính được hệ thống truy xuất đúng hay sai.

Vậy file 03 có vai trò thực tế là:

```text
Tài liệu thô dài, nhiễu, khó tìm
→ làm sạch
→ cắt thành chunk nhỏ
→ gắn metadata nguồn
→ tạo all_chunks.csv
→ chuẩn bị cho BM25/FAISS/RAG truy xuất evidence
```

File quan trọng nhất sau file 03 là:

```text
data/chunks/all_chunks.csv
```

